# 09｜独立评估统计与固定门槛

对应[第 09 章](../course/09-independent-evaluation.md)。本 Notebook 复核已提交的真实 compact evidence；它不会恢复策略，也不产生新的正式评估。

## 开始前诊断

先不要运行后文。写下你的选择，再运行下一格获得针对性反馈：

1. 964/1,024 的 Wilson 上界高于 95%，固定 95% 门槛应判什么？
2. 四个 seed 回合数不同时，aggregate 应平均四个 rate，还是加总 successes/episodes？
3. 精选的 4 成功、4 失败轨迹能否估计总体成功率？

In [ ]:
diagnostic_answers = {
    'gate': 'FAIL',
    'aggregate': 'sum counts',
    'fixture': 'behavior examples',
}
print('先保留答案；环境初始化后会逐项检查。')

## 学习目标

从 successes/episodes 重算点估计；理解 Wilson 区间；检查 per-seed 加总和 worst seed；严格执行预注册门槛；区分随机评估样本与精选轨迹样例。

## 本节知识地图

本节只学 5 个判断概念。它们回答不同问题，不能互相替代。

| 知识点 | 回答的问题 | 项目证据 | 掌握检查 |
| --- | --- | --- | --- |
| 点估计 | 这批回合实际成功了多少比例？ | successes/episodes | 手算 964/1024 |
| Wilson 区间 | 二项抽样的不确定范围多大？ | confidence interval | 解释样本量影响 |
| aggregate | 所有 seed 合并后的总体结果是什么？ | 分子分母加总 | 不盲目平均 rates |
| worst seed | 是否有某个随机批次明显较弱？ | per-seed records | 找出最小 rate |
| 固定门槛 | 按运行前规则判 PASS 还是 FAIL？ | acceptance criteria | 不用区间改判 |

## 关键概念与符号

| 名词 | 含义 | 不要混淆 |
| --- | --- | --- |
| $k/n$ | 成功数/总回合数 | 不是 reward 均值 |
| 95% Wilson CI | 对未知成功概率的区间估计 | 不是 95% 成功门槛 |
| held-out seed | 未参与选择/调参的评估随机种子 | 反复查看后会变成开发集 |
| fixture | 为教学精选的少量轨迹 | 不能用来估计总体 rate |

> **判定顺序：** 先核对协议和分母，再报点估计/区间/worst seed，最后按预注册门槛判定。

## 先预测

1. 964/1,024 是否达到固定 95% 点估计门槛？
2. 若 Wilson 上界超过 95%，工程门槛能否改判 PASS？
3. 四个 seed 分别成功 235、242、245、242，worst-seed rate 是多少？
4. 课程 fixture 有意选 4 成功、4 失败，能否报告策略成功率为 50%？

## 运行与观察

读取两类证据：compact report 保存完整 1,024 回合聚合；fixture 只保存 8 条用于学习轨迹字段的精选例子。

In [ ]:
from pathlib import Path
import math
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from course_feedback import check_choice, check_value, save_progress
from course_utils import assert_course_kernel, load_json, wilson_interval

assert_course_kernel(ROOT)
report_path = ROOT / 'reproduction/results/linux-guide-free-left-trajectory-analysis.json'
fixture_path = ROOT / 'docs/data/guide-free-left-episodes-fixture.json'
report = load_json(report_path)
fixture = load_json(fixture_path)
print('Formal evidence:', report_path.relative_to(ROOT))
print('Teaching fixture:', fixture_path.relative_to(ROOT))
check_choice('固定门槛', diagnostic_answers['gate'], 'FAIL', hint='点估计规则和区间回答不同问题。', explanation='964/1,024 的点估计低于预注册的 95% 门槛。')
check_choice('聚合方式', diagnostic_answers['aggregate'], 'sum counts', hint='先合并分子与分母。', explanation='先加总 successes 和 episodes，才能得到总体点估计。')
check_choice('fixture 用途', diagnostic_answers['fixture'], 'behavior examples', hint='精选样本不代表总体。', explanation='fixture 只用于理解字段和行为，不用于估计 rate。')

## Worked example

### 1. 聚合点估计和 Wilson 区间

点估计描述这批样本；区间描述二项抽样不确定性；门槛是运行前冻结的决策规则。三者回答不同问题。

In [ ]:
episodes = report['protocol']['episodes']
successes = report['aggregate']['successes']
failures = report['aggregate']['failures']
rate = successes / episodes
interval = wilson_interval(successes, episodes)
print(f'successes = {successes}/{episodes} = {rate:.4%}')
print(f'Wilson 95% = [{interval[0]:.4%}, {interval[1]:.4%}]')
assert successes + failures == episodes
assert np.allclose(interval, report['aggregate']['success_rate_wilson_95'], atol=1e-6)

## 故意出错

下面故意把 8 条精选轨迹当作随机成功率样本。运行后阅读反馈，并把 `fixture_claim` 改成 `behavior examples`。

In [ ]:
fixture_claim = '50% success rate'  # 故意错误
check_choice(
    '精选轨迹用途', fixture_claim, 'behavior examples',
    hint='检查数据如何抽样：4 成功、4 失败是人为挑选，不是随机抽样。',
    explanation='精选轨迹用于解释行为和字段；总体率必须来自完整随机评估。',
)

### 2. Per-seed 防止平均值隐藏弱批次

四个 held-out seed 的回合数相同，因此 aggregate 恰好也是四个 rate 的平均；一般情况下仍应按分子分母加总。

In [ ]:
per_seed = report['per_seed']
seed_labels = [str(row['seed']) for row in per_seed]
seed_rates = np.array([row['successes'] / row['episodes'] for row in per_seed])
worst_seed = float(seed_rates.min())
assert sum(row['successes'] for row in per_seed) == successes
assert sum(row['episodes'] for row in per_seed) == episodes
print('per-seed rates:', dict(zip(seed_labels, np.round(seed_rates, 4))))
print(f'worst seed = {worst_seed:.4%}')

plt.figure(figsize=(7, 3.5))
plt.bar(seed_labels, seed_rates, color='tab:blue')
plt.axhline(0.90, color='tab:green', linestyle='--', label='90% worst-seed gate')
plt.ylim(0.85, 1.0); plt.xlabel('held-out seed'); plt.ylabel('success rate')
plt.legend(); plt.title('Guide-free left evaluation by seed'); plt.show()

### 3. 样本量如何影响区间

在约 94% 点估计附近增加样本量，Wilson 区间通常收窄。62/64 看起来很高，但分母远小于 1,024。

In [ ]:
sample_sizes = np.array([32, 64, 128, 256, 512, 1024])
approx_successes = np.rint(sample_sizes * rate).astype(int)
sample_intervals = np.array([wilson_interval(int(k), int(n)) for k, n in zip(approx_successes, sample_sizes)])
widths = sample_intervals[:, 1] - sample_intervals[:, 0]
plt.figure(figsize=(7, 3.5))
plt.plot(sample_sizes, widths * 100, marker='o')
plt.xlabel('episodes'); plt.ylabel('Wilson interval width (percentage points)')
plt.title('More episodes reduce binomial uncertainty'); plt.grid(alpha=0.25); plt.show()
assert widths[-1] < widths[0]

### 4. 固定门槛不是置信区间

目标是 aggregate ≥95%、worst seed ≥90%。即使区间覆盖 95%，964/1,024 的点估计仍未达到预先规则。

In [ ]:
aggregate_target = 0.95
worst_seed_target = 0.90
required_successes = math.ceil(aggregate_target * episodes)
aggregate_passed = rate >= aggregate_target
worst_seed_passed = worst_seed >= worst_seed_target
print('aggregate:', 'PASS' if aggregate_passed else 'FAIL', f'(needs {required_successes - successes} more successes)')
print('worst seed:', 'PASS' if worst_seed_passed else 'FAIL')
print('overall:', 'PASS' if aggregate_passed and worst_seed_passed else 'FAIL')

## 动手修改

将 `hypothetical_successes` 改成 964、972、973、990。每次先预测 point-estimate gate，再运行。注意这是假设计算，不得覆盖正式 report。最后恢复 973。

In [ ]:
success_slider = widgets.IntSlider(value=973, min=900, max=1024, step=1, description='successes')
display(success_slider)
hypothetical_successes = success_slider.value
hypothetical_rate = hypothetical_successes / episodes
hypothetical_interval = wilson_interval(hypothetical_successes, episodes)
print(f'{hypothetical_successes}/{episodes} = {hypothetical_rate:.4%}')
print(f'Wilson 95% = [{hypothetical_interval[0]:.4%}, {hypothetical_interval[1]:.4%}]')
print('fixed gate:', 'PASS' if hypothetical_rate >= aggregate_target else 'FAIL')

## 自测

fixture 的 8 条记录被按每 seed 一成功一失败挑选，只用于查看字段。下面的断言故意阻止你把它当随机成功率样本。

In [ ]:
examples = fixture['curated_episode_examples']
assert fixture['fixture_role'] == 'teaching_only_curated_examples_not_a_rate_sample'
assert len(examples) == 8 and sum(row['success'] for row in examples) == 4
assert required_successes == 973 and required_successes - successes == 9
assert not aggregate_passed and worst_seed_passed
assert hypothetical_successes == required_successes and hypothetical_rate >= aggregate_target
print('PASS: aggregation, Wilson interval, per-seed audit, fixed gate, and fixture boundary')

## 项目源码连接

Notebook 中的三层判断对应正式评估报告：`aggregate` 保存合并分子/分母，`per_seed` 暴露弱批次，`acceptance_criteria` 保存预先冻结的门槛。教学 fixture 只提供可读轨迹，不替代正式 `episode_records`。

## Exit ticket

不回看前文，完成下面四个判断。全部通过才建议进入失败分析。

In [ ]:
exit_answers = {
    'estimate': 'k/n',
    'aggregate': 'sum numerator and denominator',
    'gate': 'point estimate',
    'fixture': 'not a rate sample',
}
checks = [
    check_choice('点估计', exit_answers['estimate'], 'k/n', hint='寻找成功数和总回合数。', explanation='点估计是 successes/episodes。'),
    check_choice('跨 seed 聚合', exit_answers['aggregate'], 'sum numerator and denominator', hint='不要默认各 seed 分母相同。', explanation='跨 seed 先合并分子和分母。'),
    check_choice('固定门槛依据', exit_answers['gate'], 'point estimate', hint='区间不是预注册判定规则。', explanation='本协议按点估计执行固定门槛。'),
    check_choice('精选轨迹边界', exit_answers['fixture'], 'not a rate sample', hint='检查 fixture 的挑选方式。', explanation='精选轨迹不能估计总体成功率。'),
]
if all(checks):
    print('READY：你已能区分统计估计、工程门槛与教学样本。')
SAVE_PROGRESS = False
if SAVE_PROGRESS:
    save_progress(ROOT, '09_evaluation_statistics', 'green' if all(checks) else 'yellow')

## 学完请记住

关闭本页后，你应能脱稿说出：

1. 点估计描述已观察样本，Wilson 区间描述抽样不确定性；
2. 门槛是提前冻结的工程规则，不因区间覆盖目标而改判；
3. aggregate 应加总 successes 和 episodes，不总是简单平均 rate；
4. worst seed 用于暴露平均数掩盖的弱批次；
5. 精选视频/fixture 可解释行为，不能估计成功率。

若结论没有同时写分子、分母、协议和门槛，它还不是可复核的评估结论。

## 反思与记录

用[评估结论模板](../templates/evaluation-conclusion.md)写四句话，必须包含 964/1,024、点估计、Wilson、worst seed 和固定门槛 FAIL。再说明：

- 本 Notebook 证明你会复核统计，但为什么最多是 Gate 4 PRACTICED？
- 要达到 READY，还缺自己的 checkpoint、完整 episode records、协议和 SHA-256 中的哪些证据？